Áreas Alta e Muito Alta

In [ ]:
from pathlib import Path
import sys
import pandas as pd

sys.path.append("/code/scripts")

from analysis_config import ANALYSIS, structural_pairs
from comparison_utils import (
    priority_agreement,
    priority_frequency
)

In [ ]:
pilots = ["fundao", "badajoz"]
all_summary = []

for pilot in pilots:
    pilot_dir = ANALYSIS / "02_classes" / pilot
    inventory = pd.read_csv(
        pilot_dir / f"class_inventory_{pilot}.csv"
    )

    paths = dict(zip(
        inventory["map_id"],
        inventory["local_class_raster"]
    ))

    out_dir = ANALYSIS / "05_priority_agreement" / pilot
    agreement_dir = out_dir / "agreement_rasters"
    agreement_dir.mkdir(parents=True, exist_ok=True)

    for pair in structural_pairs(pilot):
        output = agreement_dir / f"{pair['comparison_id']}_priority.tif"

        table = priority_agreement(
            paths[pair["map_a"]],
            paths[pair["map_b"]],
            output,
            pilot=pilot,
            **pair
        )

        all_summary.append(table)

    if pilot == "fundao":
        for product in ["susceptibility", "hazard"]:
            rasters = [
                paths[f"C{scenario}_{product}"]
                for scenario in range(1, 7)
            ]

            priority_frequency(
                rasters,
                out_dir / f"priority_frequency_C1_C6_{product}.tif"
            )

In [ ]:
summary = pd.concat(all_summary, ignore_index=True)
out_dir = ANALYSIS / "05_priority_agreement"
out_dir.mkdir(parents=True, exist_ok=True)
output = out_dir / "priority_agreement.xlsx"

summary.to_excel(
    output,
    sheet_name="Priority_agreement",
    index=False
)

print("Resultados guardados em:", output)
summary